# WASB fine-tune — SMOKE TEST (Kaggle)

Versión Kaggle del smoke test. Confirma que HRNet(wasb) carga los pesos soccer,
el dataloader entrega (imgs, heatmaps) sobre brasil (denso por interpolación), el
modelo produce salida y la QFL computa.

## ⚙️ ANTES DE CORRER (en el panel derecho de Kaggle):
1. **Settings → Accelerator → GPU T4 x2** (o P100).
2. **Settings → Internet → ON** (hace falta para git clone / pip / gdown).
3. **Add Input → Datasets →** subí el video `brasil_noruega.mp4` como un Dataset tuyo.
   Queda en `/kaggle/input/<nombre-del-dataset>/brasil_noruega.mp4`.

Las celdas 3-4 son diagnósticas: pegame su salida y finalizo el loop (celda 5).

## 1) Setup (repos + pesos + parches) — sin Drive

In [ ]:
import os, subprocess, glob
os.chdir('/kaggle/working')
if not os.path.exists('WASB-SBDT'):
    !git clone -q https://github.com/nttcom/WASB-SBDT.git
if not os.path.exists('ncf_event_tracker'):
    !git clone -q --branch events-model https://github.com/pipachiesa/ncf_event_tracker.git
!cd ncf_event_tracker && git pull -q origin events-model
!pip install -q hydra-core omegaconf gdown
W='/kaggle/working/wasb_soccer_best.pth.tar'
if not os.path.exists(W):
    import gdown; gdown.download(id='1pg0MpMtKZ6ziYEr4oyfKYPOO3hjLw94l', output=W, quiet=True)
# parche numpy 2.0
subprocess.run(r"grep -rl 'np\.Inf' WASB-SBDT/src | xargs -r sed -i 's/np\.Inf/np.inf/g'", shell=True)
subprocess.run(r"grep -rl 'np\.NaN' WASB-SBDT/src | xargs -r sed -i 's/np\.NaN/np.nan/g'", shell=True)
print('setup OK | peso:', os.path.exists(W))
print('datasets montados en /kaggle/input:', os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else 'NINGUNO')

## 2) Encontrar el video + construir el dataset (brasil, denso por interpolación)

In [ ]:
# busca brasil_noruega.mp4 en cualquier dataset de /kaggle/input
hits=glob.glob('/kaggle/input/**/brasil_noruega.mp4', recursive=True)
assert hits, 'FALTA el video: subí brasil_noruega.mp4 como Dataset (Add Input).'
VIDEO=hits[0]; print('video:', VIDEO)
ROOT='/kaggle/working/wasb_ft/soccer'
!cd /kaggle/working/ncf_event_tracker && python3 events_model/make_wasb_dataset.py \
    --video "{VIDEO}" \
    --labels events_model/dataset/ball_gt/brasil_noruega_ball_labels.csv \
    --out {ROOT} --clip brasil --stride 2 --max-interp 12
print('frames:', len(os.listdir(f'{ROOT}/frames/brasil')))

## 3) DIAGNÓSTICO A: cargar modelo + pesos

In [ ]:
import sys; sys.path.insert(0,'/kaggle/working/WASB-SBDT/src')
import torch, importlib
for modname in ['models','dataloaders','losses']:
    try:
        m=importlib.import_module(modname); print(f'{modname}:', [a for a in dir(m) if a.startswith("build_")])
    except Exception as e: print(f'{modname}: {type(e).__name__}: {e}')
from hydra import compose, initialize_config_dir
cfgdir='/kaggle/working/WASB-SBDT/src/configs'
has_qfl=os.path.exists(cfgdir+'/loss/qfl.yaml')
with initialize_config_dir(version_base=None, config_dir=cfgdir):
    ov=['dataset=soccer','model=wasb',f'dataset.root_dir={ROOT}',
        'dataset.train.videos=[brasil]','dataset.test.videos=[brasil]',
        'runner.gpus=[0]','detector.model_path='+W]
    if has_qfl: ov.append('loss=qfl')
    cfg=compose(config_name='eval', overrides=ov)
print('\nmodel.name:', cfg.model.name, '| loss:', cfg.get('loss'))
from models import build_model
model=build_model(cfg).cuda()
ck=torch.load(W, map_location='cpu')
res=model.load_state_dict(ck['model_state_dict'], strict=False)
print('missing:', len(res.missing_keys), '| unexpected:', len(res.unexpected_keys), '(0/0 = perfecto)')

## 4) DIAGNÓSTICO B: un batch + salida del modelo

In [ ]:
from dataloaders import build_dataloader
loaders=build_dataloader(cfg); print('loaders:', len(loaders))
train_loader=loaders[0]
batch=next(iter(train_loader))
print('batch tipos:', [type(x).__name__ for x in batch])
for i,x in enumerate(batch):
    if torch.is_tensor(x): print(f'  [{i}] shape {tuple(x.shape)} dtype {x.dtype}')
    else: print(f'  [{i}] {type(x).__name__}')
imgs=batch[0].cuda(); model.eval()
with torch.no_grad(): out=model(imgs)
print('salida tipo:', type(out).__name__)
if torch.is_tensor(out): print('  shape', tuple(out.shape))
elif isinstance(out,(list,tuple)): print('  lista de', len(out), [tuple(o.shape) if torch.is_tensor(o) else type(o).__name__ for o in out])
print('\n>>> PEGAME LA SALIDA DE 3 Y 4 -> finalizo el loop (celda 5).')

## 5) (pendiente) Loop de fine-tune — lo completo con los shapes de 3-4

In [ ]:
print('Esperando shapes de las celdas 3-4.')